<a href="https://colab.research.google.com/github/Lyv-ux/DI_Bootcamp/blob/main/W5D5_DC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# DAILY CHALLENGE: IMDB BINARY TEXT CLASSIFICATION
# COMPLETE SOLUTION IN ONE CELL
# ==========================================================

# =========================
# 1. Import Libraries
# =========================

import numpy as np
import matplotlib.pyplot as plt

from tensorflow import keras
from tensorflow.keras import models
from tensorflow.keras import layers

# =========================
# 2. Load IMDB Dataset
# Keep only the 10,000 most frequent words
# =========================

num_words = 10000

(train_data, train_labels), (test_data, test_labels) = keras.datasets.imdb.load_data(
    num_words=num_words
)

print("Training samples:", len(train_data))
print("Test samples:", len(test_data))

# =========================
# 3. Convert sequences into binary vectors
# Example:
# [3,5] -> vector of length 10000
# =========================

def vectorize_sequences(sequences, dimension=10000):

    results = np.zeros((len(sequences), dimension))

    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1

    return results

# Vectorize inputs
x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)

# Convert labels to float
y_train = np.asarray(train_labels).astype("float32")
y_test = np.asarray(test_labels).astype("float32")

print("x_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)

# =========================
# 4. Create Validation Set
# First 10,000 samples = validation
# Remaining samples = training
# =========================

x_val = x_train[:10000]
partial_x_train = x_train[10000:]

y_val = y_train[:10000]
partial_y_train = y_train[10000:]

print("Training shape:", partial_x_train.shape)
print("Validation shape:", x_val.shape)

# =========================
# 5. Build the Model
# - Two hidden layers
# - ReLU activation
# - Sigmoid output
# =========================

model = models.Sequential()

model.add(
    layers.Dense(
        16,
        activation="relu",
        input_shape=(10000,)
    )
)

model.add(
    layers.Dense(
        16,
        activation="relu"
    )
)

model.add(
    layers.Dense(
        1,
        activation="sigmoid"
    )
)

# =========================
# 6. Compile the Model
# Binary classification:
# - RMSprop
# - Binary Crossentropy
# - Accuracy
# =========================

model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# =========================
# 7. Train the Model
# 20 epochs
# Batch size = 512
# =========================

history = model.fit(
    partial_x_train,
    partial_y_train,
    epochs=20,
    batch_size=512,
    validation_data=(x_val, y_val),
    verbose=1
)

# =========================
# 8. Plot Training & Validation Loss
# =========================

history_dict = history.history

loss_values = history_dict["loss"]
val_loss_values = history_dict["val_loss"]

epochs = range(1, len(loss_values) + 1)

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)

plt.plot(
    epochs,
    loss_values,
    "bo",
    label="Training Loss"
)

plt.plot(
    epochs,
    val_loss_values,
    "b",
    label="Validation Loss"
)

plt.title("Training and Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

# =========================
# 9. Plot Training & Validation Accuracy
# =========================

acc_values = history_dict["accuracy"]
val_acc_values = history_dict["val_accuracy"]

plt.subplot(1,2,2)

plt.plot(
    epochs,
    acc_values,
    "ro",
    label="Training Accuracy"
)

plt.plot(
    epochs,
    val_acc_values,
    "r",
    label="Validation Accuracy"
)

plt.title("Training and Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

# =========================
# 10. Find Best Epoch
# Based on minimum validation loss
# =========================

best_epoch = np.argmin(val_loss_values) + 1

print("\nBest Epoch:", best_epoch)

# =========================
# 11. Retrain Model
# Using optimal number of epochs
# =========================

final_model = models.Sequential()

final_model.add(
    layers.Dense(
        16,
        activation="relu",
        input_shape=(10000,)
    )
)

final_model.add(
    layers.Dense(
        16,
        activation="relu"
    )
)

final_model.add(
    layers.Dense(
        1,
        activation="sigmoid"
    )
)

final_model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

final_model.fit(
    x_train,
    y_train,
    epochs=best_epoch,
    batch_size=512,
    verbose=1
)

# =========================
# 12. Evaluate on Test Set
# =========================

test_loss, test_accuracy = final_model.evaluate(
    x_test,
    y_test,
    verbose=0
)

# =========================
# 13. Report Final Results
# =========================

print("\n===== FINAL TEST RESULTS =====")
print("Test Loss:", round(test_loss, 4))
print("Test Accuracy:", round(test_accuracy, 4))

# =========================
# 14. Analysis
# =========================

print("\n===== ANALYSIS =====")
print("""
- Training loss should decrease steadily.

- Validation loss usually starts increasing after a few epochs.
  This is a sign of overfitting.

- The best epoch is selected using the minimum validation loss.

- Retraining with the optimal number of epochs improves
  generalization on unseen data.

- Accuracy close to 85%-90% is common for this architecture
  on the IMDB dataset.
""")